<a id="malthus"></a>

$$
\text{人口增长模型 (Population Growth Model)} \\ \quad \\
\text{1. 马尔萨斯模型 (Malthus Model, 指数增长)} \\
\text{假设: 单位时间内的出生数和死亡数都和当前人口成正比, 出生率 b, 死亡率 d 为常数} \\
P(t + \Delta t) - P(t) = b P(t) \Delta t - d P(t) \Delta t = r P(t) \Delta t, \quad r = b - d \text{ (净增长率)} \\
\frac{P(t + \Delta t) - P(t)}{\Delta t} = r P(t) \xrightarrow{\Delta t \to 0} \frac{dP}{dt} = r P \\ \quad \\
\text{分离变量: } \int \frac{dP}{P} = \int r \, dt \implies \ln P = rt + C \implies P(t) = P_0 e^{r(t - t_0)} \\ \quad \\
\text{倍增时间 (Doubling Time): } P_0 e^{rT} = 2 P_0 \implies T = \frac{\ln 2}{r} \\
\text{问题: } t \to \infty \text{ 时 } P \to \infty \text{, 没有考虑资源有限, 只适合短期或人口较少时}
$$

数学基础: [从导数建立 ODE](../math_basics/calculus.ipynb#ode) · [分离变量法](../math_basics/calculus.ipynb#ode-separable)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

# 美国人口普查数据 1790-2000 (百万人), 数学建模的经典数据
years = np.arange(1790, 2001, 10)
population = np.array([3.9, 5.3, 7.2, 9.6, 12.9, 17.1, 23.2, 31.4, 38.6, 50.2, 62.9,
                       76.0, 92.0, 105.7, 122.8, 131.7, 150.7, 179.3, 203.2, 226.5, 248.7, 281.4])
t = years - 1790

# Malthus: ln P = ln P0 + r t, 是关于 t 的直线, 用前 8 个点 (1790-1860) 做线性回归
early = slice(0, 8)
r_malthus, lnP0 = np.polyfit(t[early], np.log(population[early]), 1)
P0_malthus = np.exp(lnP0)
print(f'Malthus (1790-1860 拟合): r = {r_malthus:.5f}, P0 = {P0_malthus:.4f}, 倍增时间 = {np.log(2) / r_malthus:.2f} 年')

t_fine = np.linspace(0, 210, 300)
plt.figure(figsize=(4, 3))
plt.plot(years, population, 'ko', markersize=3, label='census')
plt.plot(1790 + t_fine, P0_malthus * np.exp(r_malthus * t_fine), label='Malthus')
plt.ylim(0, 600)
plt.ylabel('population (million)')
plot_nicely()
plt.show()
print(f'Malthus 预测 2000 年 = {P0_malthus * np.exp(r_malthus * 210):.1f}, 实际 = 281.4 (后期严重高估)')

<a id="logistic"></a>

$$
\text{2. 逻辑斯蒂模型 (Logistic Model, 阻滞增长)} \\
\text{改进: 人均增长率 } \frac{1}{P}\frac{dP}{dt} \text{ 不再是常数, 而是随着人口增加而下降 (资源, 空间有限)} \\
\text{最简单的假设: 人均增长率是 P 的线性递减函数 } r(P) = r - sP \\
\text{设环境最多能容纳 K 个人 (环境容纳量 Carrying Capacity), 达到 K 时不再增长: } r(K) = r - sK = 0 \implies s = \frac{r}{K} \\
r(P) = r\left(1 - \frac{P}{K}\right) \implies \frac{dP}{dt} = r P \left(1 - \frac{P}{K}\right) \\
\text{(P 远小于 K 时退化成 Malthus 模型; } 1 - \frac{P}{K} \text{ 是 "剩余资源的比例")}
$$

<a id="logistic-solution"></a>

$$
\text{3. 解析解: 分离变量 + 部分分式 (Partial Fractions)} \\
\frac{dP}{P(1 - P/K)} = r \, dt, \quad \frac{1}{P(1 - P/K)} = \frac{K}{P(K - P)} \\
\text{拆分: 设 } \frac{K}{P(K - P)} = \frac{A}{P} + \frac{B}{K - P} \implies A(K - P) + BP = K \\
\text{令 P = 0: } AK = K \implies A = 1; \quad \text{令 P = K: } BK = K \implies B = 1 \\
\int \left(\frac{1}{P} + \frac{1}{K - P}\right) dP = \int r \, dt \implies \ln P - \ln(K - P) = rt + C \\
\ln \frac{P}{K - P} = rt + C \implies \frac{P}{K - P} = e^{C} e^{rt} \\
\text{t = 0 代入: } e^C = \frac{P_0}{K - P_0} \text{, 解出 P:} \\
P = (K - P) \frac{P_0}{K - P_0} e^{rt} \implies P\left(1 + \frac{P_0}{K - P_0} e^{rt}\right) = K \frac{P_0}{K - P_0} e^{rt} \\
P(t) = \frac{K}{1 + a e^{-rt}}, \quad a = \frac{K - P_0}{P_0} \\
\text{这就是非线性回归里的 S 型曲线 } \hat{y} = \frac{k}{1 + a e^{-bx}} \text{, 现在知道了它是从哪个微分方程来的}
$$

相关模型: [非线性回归中的 S 型曲线](linear_regression_model/non_linear_regression.ipynb) · 数学基础: [分离变量法](../math_basics/calculus.ipynb#ode-separable)

In [ ]:
r_demo, K_demo, P0_demo = 0.03, 300.0, 3.9
logistic = lambda t, K, r, P0: K / (1 + (K - P0) / P0 * np.exp(-r * t))

# 部分分式检验
P_test = np.array([10.0, 100.0, 250.0])
print(f'K / (P (K - P)) == 1/P + 1/(K - P) ? {np.allclose(K_demo / (P_test * (K_demo - P_test)), 1 / P_test + 1 / (K_demo - P_test))}')

# 解析解 vs solve_ivp
t_eval = np.linspace(0, 400, 401)
sol = solve_ivp(lambda t, P: r_demo * P * (1 - P / K_demo), [0, 400], [P0_demo], t_eval=t_eval, rtol=1e-10, atol=1e-10)
print(f'解析解 == solve_ivp ? {np.allclose(sol.y[0], logistic(t_eval, K_demo, r_demo, P0_demo), atol=1e-6)}')

# 解析解代回方程: dP/dt 的数值差分 == r P (1 - P / K)
h = 1e-5
dP_numeric = (logistic(t_eval + h, K_demo, r_demo, P0_demo) - logistic(t_eval - h, K_demo, r_demo, P0_demo)) / (2 * h)
P_val = logistic(t_eval, K_demo, r_demo, P0_demo)
print(f'代回原方程成立? {np.allclose(dP_numeric, r_demo * P_val * (1 - P_val / K_demo), atol=1e-6)}')

<a id="logistic-properties"></a>

$$
\text{4. 性质} \\ \quad \\
\text{(1) 平衡点 (Equilibrium): } \frac{dP}{dt} = 0 \implies P = 0 \text{ 或 } P = K \\
\text{稳定性: 记 } f(P) = rP(1 - P/K) \text{, 在平衡点 } P^* \text{ 附近一阶泰勒 } \frac{d(P - P^*)}{dt} \approx f'(P^*)(P - P^*) \\
\text{偏差 } \propto e^{f'(P^*) t} \text{: } f'(P^*) < 0 \text{ 稳定 (偏差衰减), } f'(P^*) > 0 \text{ 不稳定} \\
f'(P) = r\left(1 - \frac{2P}{K}\right) \implies f'(0) = r > 0 \text{ (不稳定)}, \quad f'(K) = -r < 0 \text{ (稳定, 人口最终趋于 K)} \\ \quad \\
\text{(2) 增长最快的时刻 (拐点 Inflection Point): } \frac{dP}{dt} = f(P) \text{ 是关于 P 的开口向下的抛物线} \\
\frac{d^2 P}{dt^2} = \frac{d}{dt} f(P) = f'(P) \frac{dP}{dt} = r\left(1 - \frac{2P}{K}\right) \cdot rP\left(1 - \frac{P}{K}\right) \\
0 < P < K \text{ 时后两项为正, 所以 } \frac{d^2 P}{dt^2} = 0 \iff P = \frac{K}{2} \text{ (P < K/2 加速增长, P > K/2 减速增长)} \\
\text{时刻: } \frac{K}{1 + a e^{-rt^*}} = \frac{K}{2} \implies a e^{-r t^*} = 1 \implies t^* = \frac{\ln a}{r}, \quad \text{最大增长速度 } f(K/2) = \frac{rK}{4} \\ \quad \\
\text{(3) 应用: 最大可持续捕捞量 (Maximum Sustainable Yield)} \\
\text{每年捕捞 h: } \frac{dP}{dt} = rP(1 - \frac{P}{K}) - h \text{, 要存在正的平衡点, 需要 } h \leq \max_P f(P) = \frac{rK}{4} \text{ (把种群维持在 K/2)}
$$

数学基础: [泰勒展开 (线性化)](../math_basics/calculus.ipynb#taylor) · [极值](../math_basics/calculus.ipynb#extremum)

In [ ]:
a_demo = (K_demo - P0_demo) / P0_demo
t_star = np.log(a_demo) / r_demo
P_curve = logistic(t_eval, K_demo, r_demo, P0_demo)
dP = r_demo * P_curve * (1 - P_curve / K_demo)
print(f'拐点: 理论 t* = ln(a) / r = {t_star:.4f}, 数值上 dP/dt 最大的 t = {t_eval[np.argmax(dP)]:.0f}')
print(f'最大增长速度: 理论 rK/4 = {r_demo * K_demo / 4:.4f}, 数值 = {dP.max():.4f}, 此时 P = {P_curve[np.argmax(dP)]:.2f} (K/2 = {K_demo / 2})')

# 稳定性: 从 K 附近和 0 附近出发
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for P_start in [5, 50, 150, 280, 400, 500]:
    s = solve_ivp(lambda t, P: r_demo * P * (1 - P / K_demo), [0, 300], [P_start], t_eval=np.linspace(0, 300, 300))
    axs[0].plot(s.t, s.y[0], label=f'P0 = {P_start}')
axs[0].axhline(K_demo, color='k', linestyle='--')
axs[0].axvline(t_star, color='gray', linestyle=':')
axs[0].set_xlabel('t')
axs[0].set_ylabel('P')
P_line = np.linspace(0, K_demo * 1.1, 200)
axs[1].plot(P_line, r_demo * P_line * (1 - P_line / K_demo), label='dP/dt = f(P)')
axs[1].axhline(0, color='k', linewidth=0.8)
axs[1].axvline(K_demo / 2, color='gray', linestyle=':', label='K/2')
axs[1].set_xlabel('P')
plot_nicely()
plt.show()

# 捕捞: rP(1 - P/K) = h 的较大根是稳定平衡点 P* = K/2 (1 + sqrt(1 - 4h / (rK))); h > rK/4 时无实根, 种群灭绝
for h_catch in [1.5, 2.0, 2.5]:
    s = solve_ivp(lambda t, P: r_demo * P * (1 - P / K_demo) - h_catch, [0, 600], [K_demo],
                  events=lambda t, P: P[0], max_step=1)
    disc = 1 - 4 * h_catch / (r_demo * K_demo)
    theory = f'理论平衡点 = {K_demo / 2 * (1 + np.sqrt(disc)):.2f}' if disc >= 0 else '无平衡点'
    status = f'在 t = {s.t_events[0][0]:.1f} 灭绝' if len(s.t_events[0]) else f'600 年后 P = {s.y[0, -1]:.2f}'
    print(f'捕捞 h = {h_catch} (rK/4 = {r_demo * K_demo / 4}): {status}, {theory}')

<a id="logistic-fitting"></a>

$$
\text{5. 参数估计 (用美国人口数据)} \\ \quad \\
\text{方法 1: 差分 + 线性回归 (不需要解析解). 把导数换成差分, 人均增长率是 P 的线性函数} \\
\frac{1}{P}\frac{dP}{dt} = r - \frac{r}{K} P \quad \text{用中心差分 } \frac{dP}{dt}\Big|_{t_k} \approx \frac{P_{k+1} - P_{k-1}}{2 \Delta t} \\
\text{对 } (P_k, \frac{1}{P_k}\frac{P_{k+1} - P_{k-1}}{2\Delta t}) \text{ 做直线回归: 截距 = r, 斜率 = } -\frac{r}{K} \\ \quad \\
\text{方法 2: K 已知时线性化. 由 } \frac{K}{P} - 1 = a e^{-rt} \implies \ln\left(\frac{K}{P} - 1\right) = \ln a - rt \text{ (关于 t 的直线)} \\ \quad \\
\text{方法 3: 非线性最小二乘 (Nonlinear Least Squares), 直接最小化 } \sum_k (P_k - \frac{K}{1 + a e^{-r t_k}})^2 \\
\text{(scipy 的 curve_fit, 内部用 Levenberg-Marquardt, 是梯度下降和牛顿法的结合; 用方法 1 的结果作为初值)}
$$

数学基础: [中心差分](../math_basics/calculus.ipynb#numerical-diff) · [梯度下降](../math_basics/calculus.ipynb#gradient-descent) · [牛顿法](../math_basics/calculus.ipynb#newton) · 相关模型: [一元线性回归](linear_regression_model/linear_regression.ipynb)

In [ ]:
dt = 10
per_capita = (population[2:] - population[:-2]) / (2 * dt) / population[1:-1]
slope, intercept = np.polyfit(population[1:-1], per_capita, 1)
r_diff, K_diff = intercept, -intercept / slope
print(f'方法 1 (差分 + 线性回归): r = {r_diff:.5f}, K = {K_diff:.2f}')

# 方法 2: 用方法 1 的 K, 再对 ln(K/P - 1) 做线性回归求 r 和 a
y_lin = np.log(K_diff / population - 1)
slope2, ln_a = np.polyfit(t, y_lin, 1)
print(f'方法 2 (K 已知时线性化): r = {-slope2:.5f}, a = {np.exp(ln_a):.4f}')

# 方法 3: 非线性最小二乘
def logistic_fit(t, K, r, P0):
    return K / (1 + (K - P0) / P0 * np.exp(-r * t))

params, cov = curve_fit(logistic_fit, t, population, p0=[K_diff, r_diff, population[0]])
K_fit, r_fit, P0_fit = params
print(f'方法 3 (curve_fit): K = {K_fit:.2f}, r = {r_fit:.5f}, P0 = {P0_fit:.4f}, 参数标准误 = {np.round(np.sqrt(np.diag(cov)), 4)}')

# 独立验证 curve_fit: 自己写目标函数, 用 scipy.optimize.minimize 最小化残差平方和
from scipy.optimize import minimize
sse = lambda p: np.sum((population - logistic_fit(t, *p))**2)
res = minimize(sse, x0=[K_diff, r_diff, population[0]], method='Nelder-Mead', options={'xatol': 1e-10, 'fatol': 1e-10, 'maxiter': 20000})
print(f'minimize: K = {res.x[0]:.2f}, r = {res.x[1]:.5f}, P0 = {res.x[2]:.4f}, 和 curve_fit 一致? {np.allclose(res.x, params, rtol=1e-3)}')

pred_years = np.array([2010, 2020])
actual = np.array([308.7, 331.4])
pred = logistic_fit(pred_years - 1790, *params)
for y, p_, a_ in zip(pred_years, pred, actual):
    print(f'{y} 年: Logistic 预测 = {p_:.1f}, 实际 = {a_}, 相对误差 = {abs(p_ - a_) / a_:.2%}')

t_fine = np.linspace(0, 260, 300)
plt.figure(figsize=(4, 3))
plt.plot(years, population, 'ko', markersize=3, label='census 1790-2000')
plt.plot(pred_years, actual, 'r^', label='census 2010, 2020')
plt.plot(1790 + t_fine, logistic_fit(t_fine, *params), label=f'Logistic K={K_fit:.0f}')
plt.plot(1790 + t_fine, P0_malthus * np.exp(r_malthus * t_fine), '--', label='Malthus')
plt.ylim(0, 500)
plt.ylabel('population (million)')
plot_nicely()
plt.show()